**Pattern 1 : Deduplication**
- **Solution** : Look for three things:
1. **PartitionBy** - What makes row duplicate?
2. **Order By** - Which copy to keep? Latest based on timestamp
3. **Row Number** - Give every row inside the group row_number() and keep rn = 1 only

**Question :** 

- Duplicate Record Identification and Removal 
- A customer table has accumulated duplicate rows where the same email address was captured more than once.   
- For each customer, return every original row plus a flag is_duplicate that marks whether the row is a stale copy.   
- Within a set of rows sharing the same email, the single most recent row (by updated_at) is the one to keep and gets is_duplicate = 0; every older row for that email gets is_duplicate = 1.   
- Rows with a missing email are always treated as unique and get is_duplicate = 0.
- **Output columns:** **customer_id, email, name, phone, updated_at, is_duplicate** Order the result by customer_id

Constraint : same updated_at keep the row with smaller customer_id

In [0]:
select * from interviewready.sqlpatterns.customers;

In [0]:
with unique_rows as (
        select customer_id,
                email,
                name,
                phone,
                updated_at,
                row_number() over(partition by email order by updated_at desc, customer_id asc) as rn
                from interviewready.sqlpatterns.customers
)
select customer_id,
                email,
                name,
                phone,
                updated_at,
                CASE 
                    WHEN email IS NULL or rn = 1 THEN 0
                    ELSE 1
                END AS is_duplicate,
                rn
from unique_rows
order by customer_id;


**Pattern 2 : Running Totals**
- **Solution** : sum() gives total sum of all rows, add sum(amount) over(order by sale_date) - Now the sum remembers - every row itself and every thing above it.
- Can you reset it every month? : Add Partition by month/year/day etc in over()
- **Catch : If two rows have same date the default frame (which is range by default) adds both of them together in one jump**
    - So either bring the data to one row per day or
    - Write the frame yourself : sum(amount) over(order by sale_date rows between unbounded preceding and current row)

**Problem: Running Total for Different Genders**
- A game tracks points scored by players on successive days. For each gender, calculate total as the cumulative score through the current day, preserving the input order when multiple scores share a day. 
Return the gender, day, and cumulative total ordered by gender and day.  
- **Output columns:** gender, day, total
- **Note:** Day identifies the relevant day.   

- **Partition By :** decides who gets their own total
- **Order By :** decides the direction
- **ROWS :** make sure ties are added one by one

In [0]:
select * from interviewready.sqlpatterns.scores;

In [0]:
with gender_grp as (
    select 
        day,
        gender,
        player_name,
        score_points,
        sum(score_points) 
                over(partition by gender 
                        order by day 
                                rows between unbounded preceding and current row
                    ) as total_points
        from interviewready.sqlpatterns.scores
)
select * from 
gender_grp order by gender,day,total_points

**Pattern 3: MoM and YoY growth**
**- 3 Steps :**
1. Aggregate : every row from a month to one month sum
2. LAG : take previous month revenue
3. Math : This month - last month = growth * 100 / last month = growth%

**Two Traps :**
- January has no previous month so growth is NULL
- Keep is as NULL dont make it zero means no growth at all.
- If last month was zero division fails , so wrap the bottom number in NULLIF
- Year over year is same theory but lag(prev_month, 12) - but lag counts rows not months if one month is missing data it will mess up and query will not give any error.

**Problem: Annual Host Growth Rate (YoY)**
- Measure annual growth in new Airbnb hosts. Return signup_year, new_hosts, prev_year_hosts, and growth_pct, calculated as the percentage change from the preceding available year; 
- The first year has empty comparison values. Sort by year.   
- **Output columns: signup_year, new_hosts, prev_year_hosts, growth_pct**

In [0]:
select * from interviewready.sqlpatterns.hosts ;

In [0]:
with yearly_hosts as (
    select 
        year(signup_date) as signup_year,
        count(*) as new_hosts
        from interviewready.sqlpatterns.hosts
        group by year(signup_date)
),
host_growth as (
    select 
    *,
    lag(new_hosts) over(order by signup_year) as prev_year_hosts
    from yearly_hosts
)
select 
    signup_year,
    new_hosts,
    prev_year_hosts,
    round((new_hosts - prev_year_hosts),2)*100.0/nullif(prev_year_hosts,0) as growth_pct -- if prev_year_hosts is 0 it will return NULL
from host_growth
order by signup_year;

**Gaps and Islands :**
- Find Users who logged in 3 days in a row
- Or Find the longest Streak

**Trick :** date - row_number
- If user logins on the same day remove the duplicates first

**Problem: Consecutive Login Days (3+ Streaks)**
- Find every user login streak lasting at least three distinct consecutive calendar days. 
- Return user_id, streak_start, streak_end, and streak_days; 
- Repeated logins on one date count as one day. Sort by user and streak start.   
- **Output columns:** user_id, streak_start, streak_end, streak_days

In [0]:
select * from interviewready.sqlpatterns.logins;

In [0]:
with unique_rows as (
    select distinct user_id,login_date 
    from interviewready.sqlpatterns.logins
),
daily_login as (
    select *,
    (login_date- row_number() over(partition by user_id order by login_date)) as grp
    from unique_rows
)
select user_id, min(login_date) as streak_start ,max(login_date) as streak_end,count(*) as streak_days
 from daily_login
 group by user_id,grp
 having streak_days>=3
 order by user_id

**Pattern 5 : Sessionization**
- If a user is inactive for 30 mins, the next click is a new session. Build the sessions.
- 3 Steps:
    - for every event get the previous event time for same user
    - check the gap : datediff if it is more than 30 mins mark row as 1 otherwise 0, very first event of user is also 1
    - running sum :  
- lag() + running_sum = session_id

**Problem: Event Sessionization with Time Gaps**
- A product team wants to group each user's activity into sessions.   
- Order every user's events by time, and whenever more than 30 minutes pass between one event and the next for that user, the next event begins a new session.   
- For every session, return user_id, session_id (the session's sequence number within that user, starting at 1 and increasing in time order), session_start (timestamp of the earliest event in the session, as a string formatted YYYY-MM-DD HH24:MI:SS), session_end (timestamp of the latest event in the session, same string format), and event_count (how many events the session contains).   
- A user may have any number of sessions, and a session may contain a single event.   
- **Output columns: user_id, session_id, session_start, session_end, event_count Order the result by user_id, then session_id**

In [0]:
select * from interviewready.sqlpatterns.events;

In [0]:
with prev_sessions as (
    select *,
    lag(event_timestamp) over(partition by user_id order by event_timestamp) as prev_ts
    from interviewready.sqlpatterns.events
),
date_diff as (
    select *,
    event_timestamp - prev_ts  as time_diff
    from prev_sessions
),
sessions as (
select *,
case 
when prev_ts is NULL or time_diff > INTERVAL '30 minutes' then 1
else 0
end as flag
 from date_diff
 ),
 final_session_tbl as (
 select user_id,event_timestamp,time_diff,flag,
  sum(flag) over(partition by user_id order by event_timestamp rows between unbounded preceding and current row) as session_id
  from sessions
 )
 select user_id, session_id, min(event_timestamp) as session_start, max(event_timestamp) as session_end,count(*) as event_count
 from final_session_tbl
 group by user_id, session_id
 order by user_id,session_id


**Pattern 6 : Funnel Analysis**
- **Problem: Product Usage Funnel Analysis**
- Slack wants a one-glance adoption funnel that shows how many users reach each stage of the product journey and where they drop off. Every registered user starts at the signup stage. A user reaches activate if they have logged at least one event, engage if they have logged 5 or more events, and convert if they have logged at least one event of type purchase. Each stage is counted independently across the full user base.   
- Return one row per stage. users_count is the number of users at that stage. conversion_from_prev is that count as a percentage of the previous stage's count (and 0.0 for the first stage, signup). cumulative_conversion is that count as a percentage of the signup count.   
- **Output columns: funnel_stage, users_count, conversion_from_prev, cumulative_conversion**
- **Sort by the stage sequence: signup, activate, engage, conver**t.  


In [0]:
select * from interviewready.sqlpatterns.events_funnel;

In [0]:
select * from interviewready.sqlpatterns.users;

In [0]:
with user_stats as (
    select u.user_id,
            count(e.event_id) as event_count,
            count(e.event_id) filter (
                                        where e.event_type = 'purchase'
                                    ) as purchase_count
    from interviewready.sqlpatterns.users u
    left join interviewready.sqlpatterns.events_funnel e
        on u.user_id = e.user_id
        group by u.user_id
),
funnel as (
        select 1 as stage_order, 'signup' as funnel_stage, count(*) as users_count
        from user_stats

        UNION ALL

        select 2, 'activate', count(*) filter(where event_count >=1)
        from user_stats
        
        UNION ALL

        select 3, 'engage', count(*) filter(where event_count >=5)
        from user_stats

        UNION ALL

        select 4, 'convert', count(*) filter(where event_count >=1)
        from user_stats
),
funnel_ith_prev as (
                select *,
                lag(users_count) over(order by stage_order) as prev_count
                from funnel
)
SELECT
    funnel_stage,
    users_count,
    CASE
        WHEN stage_order = 1 THEN 0.0
        ELSE ROUND(
            users_count * 100.0 / NULLIF(prev_count, 0),
            2
            )
    END AS conversion_from_prev,
    ROUND(
        users_count * 100.0 /
        NULLIF(
            MAX(users_count) FILTER (WHERE stage_order = 1) OVER (),
            0
            ),
        2
    ) AS cumulative_conversion
FROM funnel_ith_prev
ORDER BY stage_order  

Four Steps :
1. user_stats : one row per user
2. funnel : four Selects with UNION ALL
3. funnel_with_prev : LAG brings the previous count
4. final SELECT : the two percentages

**Pattern 7: Top N per Group**
- **Problem: Get Top N Records per Group**

- A regional sales team wants products in the three highest revenue levels within each region. Sum revenue by region and product, assign equal totals the same region_rank, and keep ranks 1 through 3. Return region, product name, total revenue, and rank ordered by region and rank.   
- **Output columns: region, product_name, total_revenue, region_rank**
- **Note:** region_rank is the tie-preserving revenue position within the region. 

In [0]:
WITH product_revenue AS (
    SELECT
        region,
        product_name,
        SUM(revenue) AS total_revenue
    FROM interviewready.sqlpatterns.sales
    GROUP BY region, product_name
),
ranked_products AS (
    SELECT
        region,
        product_name,
        total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY region
            ORDER BY total_revenue DESC
        ) AS region_rank
    FROM product_revenue
)
SELECT
    region,
    product_name,
    total_revenue,
    region_rank
FROM ranked_products
WHERE region_rank <= 3

**Pattern 8 : Pivot and Cross Tab**
- **Problem: Pivot Sales by Region**
- A sales table records individual sales, each with a product name, the region where the sale happened (North, South, East, or West), and the amount. Reshape this into one row per product with four amount columns, north, south, east, and west, where each column holds that product's total sales for the matching region. If a product had no sales in a region, that column is 0.0.   
- **Output columns: product, north, south, east, west**
- Output sorted by product alphabetically.

In [0]:
select * from interviewready.sqlpatterns.sales_pivot;

In [0]:
select product, 
        round(
            coalesce( sum(amount) filter (where region = 'North'),
                        0
            ),
            2 
        ) as North,
        round(
            coalesce( sum(amount) filter (where region = 'South'),
                        0
            ),
            2 
        ) as South,
        round(
            coalesce( sum(amount) filter (where region = 'East'),
                        0
            ),
            2 
        ) as East,
        round(
            coalesce( sum(amount) filter (where region = 'West'),
                        0
            ),
            2 
        ) as West

from
interviewready.sqlpatterns.sales_pivot
group by product
order by product

CASE WHEN Solution

In [0]:
select 
product,
    SUM(CASE WHEN region = 'North' then amount else 0 end) as North,
    SUM(CASE WHEN region = 'South' then amount else 0  end) as South,
    SUM(CASE WHEN region = 'East' then amount else 0 end) as East,
    SUM(CASE WHEN region = 'West' then amount else 0 end) as West
from interviewready.sqlpatterns.sales_pivot
group by product

**Pattern 9: Self-Joins**
- **Problem: Employees Earning More Than Their Manager**
- A compensation review flags employees who out-earn the person they report to. Each employee row records who they report to and their annual salary. 
- Return the name of every employee whose salary is strictly greater than the salary of their direct manager, paired with that manager's name.
- **Output columns: employee_name, manager_name**   
- Sort the results by the employee's id in ascending order

In [0]:
select * from interviewready.sqlpatterns.employees_selfJoin

In [0]:
select
e.name as employee_name,
m.name as manager_name
from interviewready.sqlpatterns.employees_selfJoin e 
inner join interviewready.sqlpatterns.employees_selfJoin m 
    on e.manager_id = m.employee_id
where e.salary > m.salary;

In [0]:
-- case to find employees with exact similar salary
select
e1.name as employee_name,
e2.name as other_emp_name
from interviewready.sqlpatterns.employees_selfJoin e1 
        join interviewready.sqlpatterns.employees_selfJoin e2 
    on e1.salary = e2.salary
and e1.employee_id < e2.employee_id;